# SBOM-to-Audit — Stage 4 Colab Checkpoint

This notebook independently verifies Stage 4 Operational Outlier from a clean GitHub checkout and an isolated Python environment.

It preserves the checkpoint workflow used in earlier stages and packages the exact commit, release report, deterministic outputs, pilot paper assets, and environment metadata into a downloadable evidence bundle.


In [ ]:
REPO_URL = "https://github.com/richietrap/sbom_to_audit.git"
REF = "main"  # Replace with a Stage 4 branch or tag when appropriate.
print("Repository:", REPO_URL)
print("Reference:", REF)


In [ ]:
import csv
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

WORKDIR = Path("/content/sbom_to_audit")
VENV_DIR = Path("/content/sbom_to_audit_stage4_venv")
for path in (WORKDIR, VENV_DIR):
    if path.exists():
        shutil.rmtree(path)
subprocess.run(["git", "clone", "--depth", "1", "--branch", REF, REPO_URL, str(WORKDIR)], check=True)
os.chdir(WORKDIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Commit:", COMMIT)
print("Kernel Python:", sys.version)
print("Platform:", platform.platform())


In [ ]:
try:
    subprocess.run([sys.executable, "-m", "venv", str(VENV_DIR)], check=True)
except subprocess.CalledProcessError:
    subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv"], check=True)
    subprocess.run([sys.executable, "-m", "virtualenv", str(VENV_DIR)], check=True)
PYTHON = VENV_DIR / "bin" / "python"
subprocess.run([str(PYTHON), "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "--no-cache-dir", "-e", ".[dev]"], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)
print("PASS: isolated dependency integrity")


In [ ]:
RELEASE_REPORT = Path("/content/stage4_colab_release_validation.json")
subprocess.run([str(PYTHON), "scripts/release_check.py", "--report", str(RELEASE_REPORT)], check=True)
release = json.loads(RELEASE_REPORT.read_text(encoding="utf-8"))
assert release["status"] == "PASS", release["errors"]
assert len(release["deterministic_hashes"]) == 30
print("PASS: canonical release gate")
print("PASS: 30 deterministic outputs across five scenarios")


In [ ]:
OUTPUT_ROOT = Path("/content/stage4_checkpoint_outputs")
ASSET_ROOT = Path("/content/stage4_checkpoint_paper_assets")
for path in (OUTPUT_ROOT, ASSET_ROOT):
    if path.exists():
        shutil.rmtree(path)
for scenario in sorted(Path("data/scenarios").glob("*.yaml")):
    subprocess.run([str(PYTHON), "-m", "sbom_to_audit.cli", "--scenario", str(scenario), "--output-root", str(OUTPUT_ROOT)], check=True)
subprocess.run([str(PYTHON), "paper_assets/scripts/build_stage4_assets.py", "--output-root", str(OUTPUT_ROOT), "--destination", str(ASSET_ROOT)], check=True)


In [ ]:
main_pack = json.loads((OUTPUT_ROOT / "evidence_packs/operational_outlier.json").read_text())
control_pack = json.loads((OUTPUT_ROOT / "evidence_packs/operational_outlier_control.json").read_text())
with (OUTPUT_ROOT / "state_logs/operational_outlier.csv").open(newline="") as handle:
    main_states = list(csv.DictReader(handle))
with (OUTPUT_ROOT / "state_logs/operational_outlier_control.csv").open(newline="") as handle:
    control_states = list(csv.DictReader(handle))
assert [row["observed_state"] for row in main_states] == ["Monitor", "Report-Ready", "Report-Ready", "Report-Ready"]
assert [row["observed_state"] for row in control_states] == ["Monitor", "Monitor"]
assert main_pack["vulnerability_intelligence"]["cvss_base_score"] == 6.5
assert control_pack["vulnerability_intelligence"]["cvss_base_score"] == 6.5
assert main_pack["vulnerability_intelligence"]["cvss_base_severity"] == "MEDIUM"
assert main_states[1]["E_t"] == control_states[1]["E_t"]
assert main_states[1]["A_t"] == control_states[1]["A_t"]
assert main_states[1]["deadline_posture"] == control_states[1]["deadline_posture"]

import yaml
main_spec = yaml.safe_load(Path("data/scenarios/operational_outlier.yaml").read_text())
control_spec = yaml.safe_load(Path("data/scenarios/operational_outlier_control.yaml").read_text())
assert main_spec["deadline_profile"] == control_spec["deadline_profile"]

def matched_released_sources(spec):
    released = {
        artifact_id
        for event in spec["replay_events"][:2]
        for artifact_id in event["release_artifact_ids"]
    }
    return sorted(
        (item["artifact_type"], item["path"], item["timestamp"])
        for item in spec["source_catalog"]
        if item["artifact_id"] in released and item["artifact_type"] != "asset_context"
    )

assert matched_released_sources(main_spec) == matched_released_sources(control_spec)
assert float(main_states[1]["I_t"]) == 1.0
assert float(control_states[1]["I_t"]) == 0.5
assert main_states[1]["observed_state"] == "Report-Ready"
assert control_states[1]["observed_state"] == "Monitor"
assert main_pack["decision_state"]["authorized_state"] == "Report"
assert control_pack["decision_state"]["authorized_state"] is None
supplier_claims = [claim for claim in main_pack["claims"] if claim["source_artifact_id"] == "ART-OO-VEX-001" and claim.get("status") == "active"]
assert any(claim["proposition"] == "supplier_assessment_status" and claim["value"] == "under_investigation" for claim in supplier_claims)
assert not any(claim["proposition"] == "product_affectedness" for claim in supplier_claims)
print("PASS: exact-source counterfactual match")
print("PASS: operational-impact differentiation")
print("PASS: under_investigation semantics")


In [ ]:
from IPython.display import SVG, display
display(SVG(filename=str(ASSET_ROOT / "figures/operational_outlier_impact_comparison.svg")))


In [ ]:
import datetime as dt
import hashlib
import zipfile

environment = {
    "checked_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "repository": REPO_URL,
    "reference": REF,
    "commit": COMMIT,
    "kernel_python": sys.version,
    "isolated_python": subprocess.check_output([str(PYTHON), "--version"], text=True).strip(),
    "platform": platform.platform(),
    "release_status": release["status"],
    "scenarios": ["ghost_logger", "false_comfort", "false_comfort_control", "operational_outlier", "operational_outlier_control"],
}
ENV_PATH = Path("/content/stage4_colab_environment.json")
ENV_PATH.write_text(json.dumps(environment, indent=2) + "\n")
BUNDLE = Path("/content/stage4_colab_checkpoint_evidence.zip")
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(RELEASE_REPORT, RELEASE_REPORT.name)
    archive.write(ENV_PATH, ENV_PATH.name)
    for root in (OUTPUT_ROOT, ASSET_ROOT):
        for path in sorted(root.rglob("*")):
            if path.is_file():
                archive.write(path, f"{root.name}/{path.relative_to(root)}")
digest = hashlib.sha256(BUNDLE.read_bytes()).hexdigest()
print("Created:", BUNDLE)
print("Tested Git commit:", COMMIT)
print("Colab evidence bundle SHA-256:", digest)
print("Download the ZIP and preserve both values for checkpoint registration.")
